# Broiler outline-to-volume walkthrough

This notebook exposes the complete classical pipeline, including each mask stage and every volume slice. It uses the first video in `test_videos/` when available. This checkout has no sample footage, so it creates a small synthetic demonstration video as a runnable fallback.

In [1]:
from pathlib import Path
import sys, tempfile
import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from BirdSelection import FeederRelativeBirdSelector
from FeederReference import FeederReference
from SegmentationStrategy import TwoStageOtsuSegmenter
from SliceVolumeEstimator import Circular, DensityWeightModel, Elliptical, ProfileScaled, SliceVolumeEstimator
from main import process_video
import visualize

N_SLICES = 20
CROSS_SECTION_NAME = 'circular'  # circular, elliptical, or profile
LABELS_CSV = None  # optional CSV with video_name, track_id, weight_kg
video_candidates = sorted((ROOT / 'test_videos').glob('*')) if (ROOT / 'test_videos').is_dir() else []
video_candidates = [path for path in video_candidates if path.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}]
demo_temp = None
if video_candidates:
    VIDEO_PATH = video_candidates[0]
else:
    demo_temp = tempfile.TemporaryDirectory()
    VIDEO_PATH = Path(demo_temp.name) / 'synthetic_broiler_2026-11-15.mp4'
    writer = cv2.VideoWriter(str(VIDEO_PATH), cv2.VideoWriter_fourcc(*'mp4v'), 5, (320, 240))
    for frame_index in range(10):
        demo = np.full((240, 320, 3), 85, np.uint8)
        cv2.circle(demo, (160, 120), 45, (0, 128, 255), -1)
        cv2.ellipse(demo, (225 + frame_index % 3, 120), (30, 20), 0, 0, 360, (245, 245, 245), -1)
        writer.write(demo)
    writer.release()

capture = cv2.VideoCapture(str(VIDEO_PATH))
ok, frame = capture.read()
fps = capture.get(cv2.CAP_PROP_FPS)
capture.release()
if not ok:
    raise RuntimeError(f'Could not read the selected video: {VIDEO_PATH}')
print(f'Video: {VIDEO_PATH}; first frame; fps={fps:g}; n_slices={N_SLICES}; cross_section={CROSS_SECTION_NAME}')

ModuleNotFoundError: No module named 'BirdSelection'

## 1. Feeder calibration

Thirty samples are spread across the video. Orange outer-rim points are fit robustly; the debug view compares RANSAC inliers and the ellipse with the old enclosing circle. The reported scale and uncertainty are video-specific.

In [ ]:
reference = FeederReference.from_video(str(VIDEO_PATH))
print(f'px_per_cm={reference.px_per_cm:.5f}; minor/major={reference.minor_major_ratio:.4f}; uncertainty={reference.calib_uncertainty_pct:.3f}%')
visualize.plot_feeder_calibration(frame, reference)

## 2. Segmentation, step by step

The whiteness image and both Otsu thresholds are shown alongside each intermediate mask. Automatically rejected elongated components are red; the nearest feeder-relative candidate is green.

In [ ]:
segmenter = TwoStageOtsuSegmenter(reference)
segmentation = segmenter.segment(frame)
selector = FeederRelativeBirdSelector(reference)
selection = selector.select(segmentation.candidates, frame, segmentation.rejected_elongated)
bird_candidate = selection.selected
if bird_candidate is None and segmentation.candidates:
    bird_candidate = min(segmentation.candidates, key=lambda item: item.quality.get('dist_feeder_radii', float('inf')))
if bird_candidate is None:
    raise RuntimeError('No bird candidate was found in the selected frame.')
print(f't1={segmentation.threshold_stage1:.2f}; t2={segmentation.threshold_stage2:.2f}; candidates={len(segmentation.candidates)}; structures={len(segmentation.rejected_elongated)}')
visualize.plot_segmentation_steps(frame, segmentation, bird_candidate)

## 3. Quality gates

The selected candidate keeps its measurements even when a gate rejects it. This table shows contact, shape, sharpness, border status, and the final measurable decision.

In [ ]:
visualize.plot_quality_metrics(bird_candidate)

## 4. Slice figure — the key output

The same color identifies a slice in the original-frame cuts, aligned mask, width plot, reconstructed solid, and per-slice volume chart.

In [ ]:
cross_sections = {'circular': Circular(), 'elliptical': Elliptical(0.8), 'profile': ProfileScaled()}
cross_section = cross_sections[CROSS_SECTION_NAME]
rectified_frame = reference.rectify(frame)
rectified_mask = reference.rectify(bird_candidate.mask, is_mask=True)
estimator = SliceVolumeEstimator(N_SLICES, cross_section, DensityWeightModel(0.90))
volume_result = estimator.estimate(rectified_mask, reference.px_per_cm)
visualize.plot_slices(rectified_frame, rectified_mask, volume_result, reference.px_per_cm)

## 5. Slice-count sensitivity

Compare circular, elliptical, and profile-scaled estimates from 5 through 100 slices on the same rectified bird outline.

In [ ]:
visualize.plot_slice_count_sensitivity(
    rectified_mask, reference.px_per_cm,
    [Circular(), Elliptical(0.8), ProfileScaled()]
)

## 6. Whole-video run

Run the command-line pipeline on the selected video and plot the selected track. Measurable frames are blue, rejected measurements are gray, and dashed lines show per-visit medians.

In [ ]:
output_directory = Path(tempfile.mkdtemp(prefix='broiler-results-'))
FEATURES_CSV = process_video(
    VIDEO_PATH, output_directory, segmenter_name='otsu2',
    n_slices=N_SLICES, cross_section_name=CROSS_SECTION_NAME, debug_every=0
)
print(f'Feature rows: {FEATURES_CSV}')
visualize.plot_video_features(FEATURES_CSV)

## 7. Optional scale-weight validation

Set `LABELS_CSV` to a CSV containing `video_name`, `track_id`, and `weight_kg` to compare per-visit median volume with scale weight and fit density through the origin.

In [ ]:
if LABELS_CSV:
    visualize.plot_weight_validation(FEATURES_CSV, LABELS_CSV)
else:
    print('Set LABELS_CSV to enable the optional scale-weight comparison.')